In [20]:
!pip install -r requirements.txt

Processing /wheels/flash_attn-2.6.3-cp310-cp310-linux_x86_64.whl (from -r requirements.txt (line 36))
flash_attn is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 5.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 4.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 4.8 MB/s  0:00:02eta 0:00:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: transformersm━━━━━━━━━━━━━━━━━━━━━━━━  5/13 [tokenizers]
    Found existing installation: transformers 4.48.3━━━━━━━━━━  5/13 [tokenizers]
    Uninstalling transformers-4.48.3:╸━━━━━━━━━━━━━━━  8/13 [transformers]
      Successfully uninstalled transformers-4.48.3m━━━━━━━━━━━━━━━  8/13 [transformers]
  Attempting un

In [1]:
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc
from peft import PeftModel

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
def clean_memory():
    for var in ['foundation_model', 'tokenizer']:
        if var in globals():
            del globals()[var]

    if torch.cuda.is_available():
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    gc.collect()
    print('Memory is cleaned')

def print_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f'VRAM allocated {allocated}gb, reserved {reserved}gb')
    else:
        print('No cuda')


In [3]:
clean_memory()
print_memory()

Memory is cleaned
VRAM allocated 0.0gb, reserved 0.0gb


In [4]:
test_dataset = load_from_disk('test_dataset')

In [5]:
test_dataset

Dataset({
    features: ['conversations', 'source', 'score', 'openai_dialog', 'text'],
    num_rows: 297
})

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [7]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.3'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# гарантируем eos_token_id
if tokenizer.eos_token_id is None and tokenizer.eos_token is not None:
    tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids(tokenizer.eos_token)

foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                        quantization_config=bnb_config,
                                                        device_map="auto",
                                                        attn_implementation="flash_attention_2")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [8]:
print_memory()

VRAM allocated 3.85457706451416gb, reserved 3.859375gb


In [9]:
def apply_model(model, row):
    chat = []
    for i in row['openai_dialog']:
        if i['role'] == 'user':
            chat.append(i)
            break

    prompt = tokenizer.apply_chat_template(
        chat,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    gen = model.generate(**inputs,
                         max_new_tokens=1024,
                         do_sample=False,
                         return_dict_in_generate=True,
                         repetition_penalty=1.5,
                         eos_token_id=tokenizer.eos_token_id,
                         pad_token_id=tokenizer.eos_token_id,)

    prompt_len = inputs["attention_mask"].sum(dim=1).item()
    new_tokens = gen.sequences[0, prompt_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=False)
    print(answer)
    return answer
        

In [10]:
def apply_foundation_model(row):
    answer = apply_model(foundation_model, row)
    return {'foundation_model_answer': answer}

In [11]:
test_dataset.select([43]).map(apply_foundation_model)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


Photosynthesis is a biochemical reaction that takes place in plants, algae, and some bacteria to convert carbon dioxide (CO2) from the air into glucose or sugar molecules along with oxygen gas (O2). This essential biological process uses sunlight as an energy source through a series of chemical reactions involving water and chlorophyll pigments found within specialized organelles called chromoplasts/chloroplasts inside plant cells.

The overall equation for photosynthesis can be written: 6 CO₂ + 12 H₂O → C₆H₁²O⁶ (+ Energy) + 6 O₂
This means six carbon dioxiode molecule combine with twelve hydrogen atoms obtained from water plus light energy results in one glucose molecule(C_{6}H_{12}O_6), releasing six molecular oxygens back out during this conversion phase known as cellular respiration which occurs later on when these sugars are broken down by our bodies for fuel production among other processes like growth & repair mechanisms.

Photosynthesis consists mainly two stages - Light-depend

Dataset({
    features: ['conversations', 'source', 'score', 'openai_dialog', 'text', 'foundation_model_answer'],
    num_rows: 1
})

In [12]:
lora_model = PeftModel.from_pretrained(foundation_model, './peft_lab_outputs/lora_adapter_2')
lora_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralFlashAttention2(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k

In [13]:
def apply_lora_model(row):
    answer = apply_model(lora_model, row)
    return {'lora_model_answer': answer}

In [ ]:
test_dataset.select([43]).map(apply_lora_model)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
print(test_dataset.select([43])['openai_dialog'])